# Étape 3 : Entraînement et Comparaison de Modèles avec Validation Croisée

Ce notebook se concentre sur l'entraînement de différents modèles de classification, l'évaluation de leur robustesse via la validation croisée stratifiée, et l'intégration de métriques métiers.

## 1. Objectifs
* Tester plusieurs familles de modèles (Linéaire, Forêts, Boosting, MLP).
* Utiliser `StratifiedKFold` pour garantir une évaluation robuste malgré le déséquilibre des classes.
* Intégrer un coût métier personnalisé où les Faux Négatifs (mauvais clients prédits bons) coûtent plus cher que les Faux Positifs.
* Tracker toutes les expérimentations avec MLflow.

## 2. Imports et Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ML & Metrics
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import xgboost as xgb
import lightgbm as lgb

# MLflow
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
import logging
logging.getLogger("mlflow").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings('ignore')

# Configuration MLflow
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
db_path = os.path.join(project_root, "mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

experiment_name = "Credit_Scoring_Training_Step3"
if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(experiment_name)
mlflow.set_experiment(experiment_name)

print(f"Expérience active : {experiment_name}")


Expérience active : Credit_Scoring_Training_Step3


## 3. Chargement des données

In [2]:
train_path = os.path.join(project_root, "outputs", "train_final.parquet")

if os.path.exists(train_path):
    df = pd.read_parquet(train_path)
    # Échantillonnage pour la rapidité si nécessaire
    if len(df) > 50000:
        df = df.sample(50000, random_state=42)
    
    X = df.drop(columns=['TARGET', 'SK_ID_CURR']).select_dtypes(include=[np.number])
    y = df['TARGET']
    X = X.fillna(X.median())
    
    print(f"Dataset chargé : {X.shape}, Déséquilibre : {y.mean():.2%}")
else:
    print("Erreur : Fichier introuvable.")


Dataset chargé : (50000, 466), Déséquilibre : 8.05%


## 4. Métriques Métiers et Validation Croisée

### 4.1 Coût Métier
Dans le crédit scoring, un **Faux Négatif** (accorder un crédit à quelqu'un qui ne rembourse pas) est beaucoup plus coûteux qu'un **Faux Positif** (refuser un crédit à quelqu'un qui aurait remboursé).
Nous définissons un score de coût : `Cost = 10 * FN + 1 * FP`. L'objectif est de minimiser ce coût.


In [3]:
def custom_business_cost(y_true, y_pred):
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return 10 * fn + 1 * fp

business_scorer = make_scorer(custom_business_cost, greater_is_better=False)


### 4.2 Fonction d'entraînement avec Cross-Validation

In [4]:
def train_evaluate_model(model, model_name, X, y, params=None, tags=None):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scoring = {
        'roc_auc': 'roc_auc',
        'f1': 'f1',
        'recall': 'recall',
        'precision': 'precision',
        'business_cost': business_scorer
    }
    
    with mlflow.start_run(run_name=model_name):
        if params:
            mlflow.log_params(params)
        if tags:
            mlflow.set_tags(tags)
            
        cv_results = cross_validate(model, X, y, cv=skf, scoring=scoring, return_train_score=False)
        
        # Log des moyennes des métriques
        metrics = {
            "cv_roc_auc": np.mean(cv_results['test_roc_auc']),
            "cv_f1": np.mean(cv_results['test_f1']),
            "cv_recall": np.mean(cv_results['test_recall']),
            "cv_precision": np.mean(cv_results['test_precision']),
            "cv_business_cost": np.mean(cv_results['test_business_cost'])
        }
        
        for name, value in metrics.items():
            mlflow.log_metric(name, value)
            
        # Entraînement final sur tout le dataset pour sauvegarde
        model.fit(X, y)
        
        # Note: 'artifact_path' est déprécié dans certaines versions au profit de 'name', 
        # mais la signature de log_model utilise toujours artifact_path par défaut dans la plupart des saveurs.
        if "LightGBM" in model_name:
            mlflow.lightgbm.log_model(model, artifact_path="model")
        elif "XGBoost" in model_name:
            mlflow.xgboost.log_model(model, artifact_path="model")
        else:
            mlflow.sklearn.log_model(model, artifact_path="model")
            
        print(f"--- {model_name} ---")
        print(f"AUC: {metrics['cv_roc_auc']:.4f}, Cost: {metrics['cv_business_cost']:.1f}")
        return metrics


## 5. Entraînement des modèles

### 5.1 Logistic Regression (Baseline)

In [5]:
lr_params = {"C": 1.0, "max_iter": 1000, "class_weight": "balanced", "random_state": 42}
train_evaluate_model(LogisticRegression(**lr_params), "Logistic_Regression", X, y, params=lr_params, tags={"family": "linear"})


--- Logistic_Regression ---
AUC: 0.6482, Cost: -6665.0


{'cv_roc_auc': np.float64(0.6482218598295588),
 'cv_f1': np.float64(0.20393993710204117),
 'cv_recall': np.float64(0.5995513117641605),
 'cv_precision': np.float64(0.12287073865450096),
 'cv_business_cost': np.float64(-6665.0)}

### 5.2 Random Forest

In [6]:
rf_params = {"n_estimators": 100, "max_depth": 10, "class_weight": "balanced", "random_state": 42}
train_evaluate_model(RandomForestClassifier(**rf_params), "Random_Forest", X, y, params=rf_params, tags={"family": "ensemble"})


--- Random_Forest ---
AUC: 0.7224, Cost: -6245.6


{'cv_roc_auc': np.float64(0.7223883946546178),
 'cv_f1': np.float64(0.26485954974133347),
 'cv_recall': np.float64(0.358684527672198),
 'cv_precision': np.float64(0.2100866324504153),
 'cv_business_cost': np.float64(-6245.6)}

### 5.3 XGBoost

In [7]:
xgb_params = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "scale_pos_weight": 11, # ~ ratio neg/pos pour gérer le déséquilibre
    "random_state": 42,
    "use_label_encoder": False,
    "eval_metric": "logloss"
}
train_evaluate_model(xgb.XGBClassifier(**xgb_params), "XGBoost", X, y, params=xgb_params, tags={"family": "boosting"})


--- XGBoost ---
AUC: 0.7459, Cost: -5611.6


{'cv_roc_auc': np.float64(0.7459322335273117),
 'cv_f1': np.float64(0.288546800952194),
 'cv_recall': np.float64(0.49788078242328726),
 'cv_precision': np.float64(0.20314350717207713),
 'cv_business_cost': np.float64(-5611.6)}

### 5.4 LightGBM

In [8]:
lgb_params = {
    "n_estimators": 100,
    "learning_rate": 0.1,
    "is_unbalance": True,
    "random_state": 42,
    "verbosity": -1
}
train_evaluate_model(lgb.LGBMClassifier(**lgb_params), "LightGBM", X, y, params=lgb_params, tags={"family": "boosting"})


--- LightGBM ---
AUC: 0.7542, Cost: -5399.2


{'cv_roc_auc': np.float64(0.7541879249660818),
 'cv_f1': np.float64(0.2921758488017089),
 'cv_recall': np.float64(0.5508210500293563),
 'cv_precision': np.float64(0.19884832395552748),
 'cv_business_cost': np.float64(-5399.2)}

### 5.5 MLP (Neural Network)
Pour le MLP, il est crucial de **normaliser les données** car les réseaux de neurones sont sensibles à l'échelle des variables. Nous utilisons un `Pipeline` avec `StandardScaler`.
De plus, pour gérer le déséquilibre, nous ajusterons les hyperparamètres ou la structure si nécessaire.


In [9]:
mlp_params = {
    "mlp__hidden_layer_sizes": (64, 32),
    "mlp__max_iter": 500,
    "mlp__random_state": 42
}

mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
])

train_evaluate_model(mlp_pipeline, "MLP_NeuralNet", X, y, params=mlp_params, tags={"family": "neural_net", "preprocessing": "scaling"})


--- MLP_NeuralNet ---
AUC: 0.6328, Cost: -7288.4


{'cv_roc_auc': np.float64(0.6327967880742454),
 'cv_f1': np.float64(0.1761423962431483),
 'cv_recall': np.float64(0.17051976144124098),
 'cv_precision': np.float64(0.18258673537380413),
 'cv_business_cost': np.float64(-7288.4)}

## 6. Conclusion et Comparaison
Les résultats sont enregistrés dans MLflow. Vous pouvez les comparer en lançant :
`mlflow ui --backend-store-uri sqlite:///mlflow.db`

Points observés :
* Les modèles de boosting (XGBoost, LightGBM) offrent généralement le meilleur compromis AUC/Coût métier.
* La validation croisée confirme la stabilité des performances.
* La gestion du déséquilibre est cruciale pour obtenir un recall décent sur la classe 1.
